### Imports and Connection

In [1]:
import duckdb
import pandas as pdb

con = duckdb.connect("../data/processed/recipeiq.duckdb", read_only=True)
print("Connected to RecipeIQ warehouse ✓")

recipe_count = con.execute("SELECT COUNT(*) FROM recipes").fetchone()[0]
review_count = con.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]

print(f"Recipes: {recipe_count:,}")
print(f"Reviews: {review_count:,}")

Connected to RecipeIQ warehouse ✓
Recipes: 522,517
Reviews: 1,401,768


### Top 10 Recipe Categories by Count

In [2]:
df = con.execute("""
    SELECT
        RecipeCategory,
        COUNT(*) AS recipe_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
    FROM recipes
    WHERE RecipeCategory IS NOT NULL
    GROUP BY RecipeCategory
    ORDER BY recipe_count DESC
    LIMIT 10
""").fetchdf()

print("Top 10 Recipe Categories")
print("=" * 50)
print(df.to_string(index=False))

Top 10 Recipe Categories
RecipeCategory  recipe_count  percentage
       Dessert         62072        11.9
  Lunch/Snacks         32586         6.2
 One Dish Meal         31345         6.0
     Vegetable         27231         5.2
     Breakfast         21101         4.0
     Beverages         16076         3.1
       Chicken         13249         2.5
          Meat         13131         2.5
        Breads         12804         2.5
          Pork         12603         2.4


### Average Rating By Complexity

In [3]:
df = con.execute("""
    SELECT
        CASE complexity_score
            WHEN 1 THEN '🟢 Simple'
            WHEN 2 THEN '🟡 Medium'
            WHEN 3 THEN '🔴 Complex'
        END AS difficulty,
        COUNT(*) AS recipe_count,
        ROUND(AVG(AggregatedRating), 2) AS avg_rating,
        ROUND(AVG(ReviewCount), 1) AS avg_reviews,
        ROUND(AVG(calories_per_serving), 0) AS avg_cal_per_serving
    FROM recipes
    WHERE AggregatedRating IS NOT NULL
    GROUP BY complexity_score
    ORDER BY complexity_score
""").fetchdf()

print("Rating by Complexity")
print("=" * 50)
print(df.to_string(index=False))

Rating by Complexity
difficulty  recipe_count  avg_rating  avg_reviews  avg_cal_per_serving
  🟢 Simple         35779        4.66          4.3                107.0
  🟡 Medium        171324        4.62          5.4                 93.0
 🔴 Complex         62191        4.64          5.8                101.0


### Nutrition Profile — Healthiest vs Unhealthiest Categories

In [4]:
df = con.execute ("""
    SELECT
        RecipeCategory,
        COUNT(*) AS num,
        ROUND(AVG(calories_per_serving), 0) AS avg_cal,
        ROUND(AVG(protein_per_serving), 1) AS avg_protein,
        ROUND(AVG(fat_per_serving), 1) AS avg_fat,
        ROUND(AVG(sugar_per_serving), 1) AS avg_sugar
    FROM recipes
    WHERE RecipeCategory IS NOT NULL
    AND calories_per_serving IS NOT NULL
    AND calories_per_serving < 2000 -- Exclude likely data errors
    GROUP BY RecipeCategory
    HAVING COUNT(*) >= 100 -- Only categories with enough data
    ORDER BY avg_cal ASC
    LIMIT 15

""").fetchdf()

print("Healthiest Recipe Categories (by calories)")
print("=" * 60)
print(df.to_string(index=False))

Healthiest Recipe Categories (by calories)
  RecipeCategory  num  avg_cal  avg_protein  avg_fat  avg_sugar
For Large Groups  531     11.0          0.4      0.6        0.4
      Bar Cookie 3816     19.0          0.3      1.0        1.6
    Drop Cookies 2820     24.0          0.4      1.2        1.9
    Quick Breads 5713     29.0          0.6      1.2        1.7
         Gelatin  922     36.0          0.8      1.5        4.2
           Candy 2370     36.0          0.4      1.5        4.4
          Scones  709     40.0          0.9      1.8        1.4
         Berries  249     40.0          0.9      1.5        4.4
    Yeast Breads 2531     42.0          1.1      1.1        1.0
  Punch Beverage 1315     42.0          0.3      0.4        5.2
          Melons  214     44.0          1.2      1.5        5.6
    Thanksgiving  138     45.0          1.5      2.3        1.4
         Spreads 2164     45.0          1.6      3.2        0.8
        Chutneys  174     47.0          0.9      1.2        6

### Rating Distribution — Are Most Recipes Highly Rated?

In [5]:
df = con.execute("""
    SELECT
        ROUND(AggregatedRating, 0) AS rating_bucket,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
    FROM recipes
    WHERE AggregatedRating IS NOT NULL
    GROUP BY rating_bucket
    ORDER BY rating_bucket
""").fetchdf()

print("Rating Distribution")
print("=" * 50)
print(df.to_string(index=False))

Rating Distribution
 rating_bucket  count  percentage
           1.0   1677         0.6
           2.0   2125         0.8
           3.0   9839         3.7
           4.0  46807        17.4
           5.0 208846        77.6


### Most Reviewed Recipes — The "Greatest Hits

In [6]:
df = con.execute("""
    SELECT
        Name,
        RecipeCategory,
        CAST(ReviewCount AS INTEGER) AS reviews,
        AggregatedRating AS rating,
        ingredient_count,
        ROUND(calories_per_serving, 0) AS cal_per_serving,
        CASE complexity_score
            WHEN 1 THEN 'Simple'
            WHEN 2 THEN 'Medium'
            WHEN 3 THEN 'Complex'
        END AS difficulty
    FROM recipes
    WHERE ReviewCount IS NOT NULL
    ORDER BY ReviewCount DESC
    LIMIT 20
""").fetchdf()

print("Top 20 Most Reviewed Recipes")
print("=" * 70)
print(df.to_string(index=False))

Top 20 Most Reviewed Recipes
                                                      Name      RecipeCategory  reviews  rating  ingredient_count  cal_per_serving difficulty
                                           Bourbon Chicken      Chicken Breast     3063     5.0                10            130.0     Medium
                                         Best Banana Bread        Quick Breads     2273     5.0                 8             27.0     Medium
                                To Die for Crock Pot Roast       One Dish Meal     1692     5.0                 2             37.0     Medium
         Crock-Pot Chicken With Black Beans & Cream Cheese       One Dish Meal     1657     4.5                 5            170.0     Medium
                                Creamy Cajun Chicken Pasta      Chicken Breast     1586     5.0                12            360.0     Medium
                                    Oatmeal Raisin Cookies        Drop Cookies     1410     5.0                11      

### Review Length vs Rating — Do Angry People Write More?

In [7]:
df = con.execute("""
    SELECT
        Rating AS stars,
        COUNT(*) AS review_count,
        ROUND(AVG(review_length), 0) AS avg_chars,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY review_length), 0) AS median_chars
    FROM reviews
    GROUP BY Rating
    ORDER BY Rating
""").fetchdf()

print("Review Length by Rating")
print("=" * 50)
print(df.to_string(index=False))

Review Length by Rating
 stars  review_count  avg_chars  median_chars
     0         76248      248.0         194.0
     1         16555      253.0         200.0
     2         17596      278.0         228.0
     3         50276      296.0         247.0
     4        229181      288.0         246.0
     5       1011912      274.0         235.0


### Recipe Publishing Trends Over Time

In [8]:
df = con.execute("""
    SELECT
        EXTRACT(YEAR FROM DatePublished) AS year,
        COUNT(*) AS recipes_published,
        ROUND(AVG(AggregatedRating), 2) AS avg_rating
    FROM recipes
    WHERE EXTRACT(YEAR FROM DatePublished) BETWEEN 1999 AND 2023
    GROUP BY year
    ORDER BY year
""").fetchdf()

print("Recipes Published Per Year")
print("=" * 50)
print(df.to_string(index=False))

Recipes Published Per Year
 year  recipes_published  avg_rating
 1999               3673        4.48
 2000               1886        4.50
 2001               7714        4.57
 2002              32456        4.63
 2003              29372        4.64
 2004              26686        4.64
 2005              41002        4.64
 2006              51409        4.61
 2007              70204        4.61
 2008              69485        4.64
 2009              58036        4.65
 2010              37375        4.66
 2011              24990        4.70
 2012              20896        4.69
 2013              18203        4.68
 2014               8640        4.52
 2015               4845        4.63
 2016               4269        4.58
 2017               4998        4.56
 2018               3035        4.53
 2019               1789        4.71
 2020               1554        4.89


### Ingredient Popularity — What Goes Into Most Recipes?

In [9]:
df = con.execute("""
    SELECT
        ingredient_count,
        COUNT(*) AS recipe_count,
        ROUND(AVG(AggregatedRating), 2) AS avg_rating,
        ROUND(AVG(calories_per_serving), 0) AS avg_cal
    FROM recipes
    WHERE AggregatedRating IS NOT NULL
      AND calories_per_serving IS NOT NULL
    GROUP BY ingredient_count
    HAVING COUNT(*) >= 50
    ORDER BY ingredient_count
""").fetchdf()

print("Recipes by Ingredient Count")
print("=" * 60)
print(df.to_string(index=False))

Recipes by Ingredient Count
 ingredient_count  recipe_count  avg_rating  avg_cal
                0           715        4.71    108.0
                1          2932        4.68    103.0
                2          6552        4.66    116.0
                3         10546        4.65    108.0
                4         13777        4.63    104.0
                5         16336        4.63     97.0
                6         17819        4.62     99.0
                7         18172        4.61     91.0
                8         17501        4.61     88.0
                9         15886        4.62     90.0
               10         13505        4.63     94.0
               11         10576        4.62     96.0
               12          8167        4.62     93.0
               13          5989        4.63     95.0
               14          4333        4.64     92.0
               15          3063        4.64    100.0
               16          2127        4.64    105.0
               17 

### Power Users — Who Reviews the Most?

In [10]:
df = con.execute("""
    SELECT
        AuthorName,
        COUNT(*) AS review_count,
        ROUND(AVG(Rating), 2) AS avg_rating,
        ROUND(AVG(review_length), 0) AS avg_review_length
    FROM reviews
    GROUP BY AuthorId, AuthorName
    ORDER BY review_count DESC
    LIMIT 15
""").fetchdf()

print("Top 15 Reviewers")
print("=" * 60)
print(df.to_string(index=False))

Top 15 Reviewers
         AuthorName  review_count  avg_rating  avg_review_length
        Sydney Mike          8842        4.97              359.0
          Sharon123          6605        4.80              154.0
           Boomette          5438        4.75              221.0
          Baby Kato          4693        4.91              340.0
            Annacia          4586        4.69              394.0
Kittencalrecipezazz          3963        4.94              272.0
           Rita1652          3743        4.64              259.0
            Parsley          3688        4.82              290.0
             PaulaG          3590        4.71              291.0
             lazyme          3543        4.89              209.0
         breezermom          3340        4.81              313.0
              Bergy          3260        4.79              334.0
          bmcnichol          3167        4.25              249.0
            loof751          3014        4.67              258.0
        

In [13]:
con.close()
print("\nConnection closed ✓")
print("Analysis complete ✓")


Connection closed ✓
Analysis complete ✓
